In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import re
import os
import glob
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2

## Importing labels and images into variables

In [8]:
home = "images"
directories = "train", "test", "val"

In [9]:
def read_file_to_list(file_path):
    numbers_list = []
    with open(file_path, 'r') as file:
        for line in file:
            number = int(line.strip())
            numbers_list.append(number)
        return numbers_list

In [10]:
y_train = read_file_to_list(f'{home}/{directories[0]}.txt')
y_test = read_file_to_list(f'{home}/{directories[1]}.txt')
y_val = read_file_to_list(f'{home}/{directories[2]}.txt')

In [11]:
train_folder = os.path.join(home, directories[0])
test_folder = os.path.join(home, directories[1])
val_folder = os.path.join(home, directories[2])

def get_image_content(folder_path):
    image_contents = []

    def extract_number(filename):
        match = re.search(r'\d+', filename)
        return int(match.group()) if match else -1

    filenames = sorted(os.listdir(folder_path), key=extract_number)

    for filename in filenames:
        file_path = os.path.join(folder_path, filename)
        if os.path.isfile(file_path) and filename.endswith('.png'):
            image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            image_contents.append(image)

    return image_contents

x_train = get_image_content(train_folder)
x_test = get_image_content(test_folder)
x_val = get_image_content(val_folder)

## Training the CNN

In [12]:
x_train = np.array(x_train).reshape((-1, 224, 224, 1)).astype('float32')
x_val = np.array(x_val).reshape((-1, 224, 224, 1)).astype('float32')
x_test = np.array(x_test).reshape((-1, 224, 224, 1)).astype('float32')

y_train = tf.keras.utils.to_categorical(np.array(y_train), num_classes=4)
y_val = tf.keras.utils.to_categorical(np.array(y_val), num_classes=4)
y_test = tf.keras.utils.to_categorical(np.array(y_test), num_classes=4)

In [9]:
tf.random.set_seed(42)
model = tf.keras.Sequential()
input_shape = (224, 224, 1)
model.add(tf.keras.layers.Conv2D(10, 3, activation="relu", input_shape=input_shape))
model.add(tf.keras.layers.MaxPool2D())
model.add(tf.keras.layers.Conv2D(10, 3, activation="relu"))
model.add(tf.keras.layers.MaxPool2D())
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(4, activation="softmax"))

model.compile(loss=tf.keras.losses.CategoricalCrossentropy(),
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

history = model.fit(x=x_train, y=y_train, epochs=10, validation_data=(x_val, y_val), batch_size=16)

test_loss, test_acc = model.evaluate(x_test, y_test)

Epoch 1/10
211/211 [==============================] - 15s 68ms/step - loss: 9.1317 - accuracy: 0.5461 - val_loss: 0.8279 - val_accuracy: 0.6611
Epoch 2/10
211/211 [==============================] - 15s 69ms/step - loss: 0.7658 - accuracy: 0.6792 - val_loss: 0.7016 - val_accuracy: 0.7180
Epoch 3/10
211/211 [==============================] - 14s 66ms/step - loss: 0.7032 - accuracy: 0.7000 - val_loss: 0.7111 - val_accuracy: 0.7133
Epoch 4/10
211/211 [==============================] - 14s 66ms/step - loss: 0.5681 - accuracy: 0.7753 - val_loss: 0.5550 - val_accuracy: 0.7796
Epoch 5/10
211/211 [==============================] - 14s 67ms/step - loss: 0.5163 - accuracy: 0.7978 - val_loss: 0.5549 - val_accuracy: 0.8009
Epoch 6/10
211/211 [==============================] - 14s 67ms/step - loss: 0.4603 - accuracy: 0.8153 - val_loss: 0.5085 - val_accuracy: 0.7915
Epoch 7/10
211/211 [==============================] - 15s 70ms/step - loss: 0.4262 - accuracy: 0.8361 - val_loss: 0.5228 - val_accuracy:

In [14]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [17]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
NUM_CLASSES = 4  # Liczba klas w Twoich danych

In [13]:
x_train_rgb = tf.image.grayscale_to_rgb(tf.convert_to_tensor(x_train))
x_val_rgb = tf.image.grayscale_to_rgb(tf.convert_to_tensor(x_val))
x_test_rgb = tf.image.grayscale_to_rgb(tf.convert_to_tensor(x_test))

x_train_rgb = x_train_rgb / 255.0
x_val_rgb = x_val_rgb / 255.0
x_test_rgb = x_test_rgb / 255.0

In [20]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs=predictions)
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train_rgb, y_train, validation_data=(x_val_rgb, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE)

loss, accuracy = model.evaluate(x_test_rgb, y_test)
print(f'Test accuracy: {accuracy}')

Epoch 1/10
106/106 [==============================] - 38s 346ms/step - loss: 0.7593 - accuracy: 0.7272 - val_loss: 0.5225 - val_accuracy: 0.7938
Epoch 2/10
106/106 [==============================] - 36s 344ms/step - loss: 0.4852 - accuracy: 0.8052 - val_loss: 0.4273 - val_accuracy: 0.8483
Epoch 3/10
106/106 [==============================] - 36s 344ms/step - loss: 0.3854 - accuracy: 0.8553 - val_loss: 0.4184 - val_accuracy: 0.8412
Epoch 4/10
106/106 [==============================] - 38s 356ms/step - loss: 0.3359 - accuracy: 0.8776 - val_loss: 0.4484 - val_accuracy: 0.8318
Epoch 5/10
106/106 [==============================] - 39s 365ms/step - loss: 0.3177 - accuracy: 0.8776 - val_loss: 0.3897 - val_accuracy: 0.8626
Epoch 6/10
106/106 [==============================] - 36s 342ms/step - loss: 0.2461 - accuracy: 0.9051 - val_loss: 0.3955 - val_accuracy: 0.8460
Epoch 7/10
106/106 [==============================] - 37s 346ms/step - loss: 0.2350 - accuracy: 0.9096 - val_loss: 0.4468 - val_ac